In [2]:
import cv2
import mediapipe as mp
import json
import os

print("✅ 라이브러리 임포트 완료")
print(f"MediaPipe version: {mp.__version__}")

✅ 라이브러리 임포트 완료
MediaPipe version: 0.10.33


In [3]:
# ✅ 경로 설정
VIDEO_PATH = r"D:\kpop-dance-dtw-analysis\dtw-analysis\data\videos\ReferenceVideo-love-dive.mp4"
OUTPUT_PATH = r"D:\kpop-dance-dtw-analysis\dtw-analysis\data\json\expert_lovedive.json"

# 경로 확인
print(f"영상 파일 존재 여부: {os.path.exists(VIDEO_PATH)}")
print(f"출력 폴더 존재 여부: {os.path.exists(os.path.dirname(OUTPUT_PATH))}")

영상 파일 존재 여부: True
출력 폴더 존재 여부: True


In [5]:
# 최신 MediaPipe API 방식
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import mediapipe as mp

# 최신 버전 호환 방식으로 Pose 초기화
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

print("✅ 최신 MediaPipe API 로드 완료")

✅ 최신 MediaPipe API 로드 완료


In [6]:
import urllib.request

MODEL_PATH = r"D:\kpop-dance-dtw-analysis\dtw-analysis\data\pose_landmarker_heavy.task"

# 모델 다운로드 (heavy = 정확도 최우선)
url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task"

if not os.path.exists(MODEL_PATH):
    print("📥 모델 다운로드 중... (약 30초 소요)")
    urllib.request.urlretrieve(url, MODEL_PATH)
    print("✅ 모델 다운로드 완료!")
else:
    print("✅ 모델 이미 존재함, 스킵!")

📥 모델 다운로드 중... (약 30초 소요)
✅ 모델 다운로드 완료!


In [7]:
def extract_keypoints(video_path, output_path, model_path):
    
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"🎬 영상 정보: {fps}fps / 총 {total}프레임")

    all_frames = []
    frame_idx = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int((frame_idx / fps) * 1000)

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            frame_data = {
                "frame_index": frame_idx,
                "timestamp_ms": timestamp_ms,
                "landmarks": []
            }

            if result.pose_landmarks:
                for idx, lm in enumerate(result.pose_landmarks[0]):
                    frame_data["landmarks"].append({
                        "id": idx,
                        "x": round(lm.x, 6),
                        "y": round(lm.y, 6),
                        "z": round(lm.z, 6),
                        "visibility": round(lm.visibility, 6)  # ✅ float 그대로
                    })
            else:
                frame_data["landmarks"] = None

            all_frames.append(frame_data)
            frame_idx += 1

            if frame_idx % 100 == 0:
                print(f"  처리 중... {frame_idx}/{total} 프레임")

    cap.release()

    output = {
        "metadata": {
            "source": os.path.basename(video_path),
            "fps": fps,
            "total_frames": frame_idx
        },
        "frames": all_frames
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"\n✅ 추출 완료: 총 {frame_idx}프레임 → {output_path}")
    return output

print("✅ 함수 정의 완료")

✅ 함수 정의 완료


In [8]:
data = extract_keypoints(VIDEO_PATH, OUTPUT_PATH, MODEL_PATH)

# 결과 샘플 확인 (처음 5개 관절)
sample = data["frames"][0]["landmarks"]
if sample:
    print("\n📊 첫 번째 프레임 샘플 (관절 0~4):")
    for lm in sample[:5]:
        print(f"  id={lm['id']} | x={lm['x']} | y={lm['y']} | visibility={lm['visibility']:.4f}")

🎬 영상 정보: 24.0fps / 총 4369프레임
  처리 중... 100/4369 프레임
  처리 중... 200/4369 프레임
  처리 중... 300/4369 프레임
  처리 중... 400/4369 프레임
  처리 중... 500/4369 프레임
  처리 중... 600/4369 프레임
  처리 중... 700/4369 프레임
  처리 중... 800/4369 프레임
  처리 중... 900/4369 프레임
  처리 중... 1000/4369 프레임
  처리 중... 1100/4369 프레임
  처리 중... 1200/4369 프레임
  처리 중... 1300/4369 프레임
  처리 중... 1400/4369 프레임
  처리 중... 1500/4369 프레임
  처리 중... 1600/4369 프레임
  처리 중... 1700/4369 프레임
  처리 중... 1800/4369 프레임
  처리 중... 1900/4369 프레임
  처리 중... 2000/4369 프레임
  처리 중... 2100/4369 프레임
  처리 중... 2200/4369 프레임
  처리 중... 2300/4369 프레임
  처리 중... 2400/4369 프레임
  처리 중... 2500/4369 프레임
  처리 중... 2600/4369 프레임
  처리 중... 2700/4369 프레임
  처리 중... 2800/4369 프레임
  처리 중... 2900/4369 프레임
  처리 중... 3000/4369 프레임
  처리 중... 3100/4369 프레임
  처리 중... 3200/4369 프레임
  처리 중... 3300/4369 프레임
  처리 중... 3400/4369 프레임
  처리 중... 3500/4369 프레임
  처리 중... 3600/4369 프레임
  처리 중... 3700/4369 프레임
  처리 중... 3800/4369 프레임
  처리 중... 3900/4369 프레임
  처리 중... 4000/4369 프레임
  처리 중... 4100/4369 